# Module 2: Convolutions & Image Processing

**Learning Objectives:**
- Implement 1D and 2D convolution from scratch and understand the sliding-kernel mechanism
- Master the output size formula and its parameters: padding, stride, dilation
- Understand transposed convolutions, their artifacts, and alternatives for upsampling
- Build depthwise separable convolutions and reason about parameter efficiency
- Implement BatchNorm and GroupNorm from scratch; understand why diffusion models prefer GroupNorm
- Build residual blocks with proper gradient flow -- the core building block of diffusion U-Nets

**Key Papers:**
- [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385) -- He et al. 2015
- [Group Normalization](https://arxiv.org/abs/1803.08494) -- Wu & He 2018

**Estimated Time:** 2--3 hours

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
from typing import Optional, Tuple, Union
import time

torch.manual_seed(42)
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

---
## 2.1 -- 1D and 2D Convolution from Scratch

### What convolution does

A convolution slides a small **kernel** (filter) across an input signal, computing a dot product at each position. This is the fundamental operation in all image-based neural networks, including diffusion U-Nets.

### Convolution vs correlation

Mathematically, **convolution** flips the kernel before sliding; **correlation** does not. In deep learning, we always use correlation (no flip) but call it "convolution" by convention. The network learns the kernel weights anyway, so flipping is irrelevant.

### Why convolutions work for images

| Property | Meaning | Benefit |
|----------|---------|--------|
| **Translation equivariance** | Same kernel applied everywhere | A feature detected at one location is detected everywhere |
| **Parameter sharing** | One kernel reused across all spatial positions | Dramatically fewer parameters than fully-connected layers |
| **Local receptive fields** | Each output depends only on a small input region | Exploits spatial locality in images |

### Multi-channel convolution

For a layer with `C_in` input channels and `C_out` output channels using a `K_h x K_w` kernel, the weight tensor has shape `(C_out, C_in, K_h, K_w)`. Each output channel is a sum of convolutions across all input channels.

### Worked Example: 1D Convolution

In [ ]:
def conv1d_loop(x: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    """1D convolution (correlation) using an explicit loop.
    
    Args:
        x: Input signal of shape (L,)
        kernel: Filter of shape (K,)
    
    Returns:
        Output of shape (L - K + 1,)
    """
    L = x.shape[0]
    K = kernel.shape[0]
    out_len = L - K + 1
    output = torch.zeros(out_len)
    for i in range(out_len):
        output[i] = torch.dot(x[i:i+K], kernel)
    return output


def conv1d_vectorized(x: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    """1D convolution (correlation) using unfolded views -- no loops.
    
    Args:
        x: Input signal of shape (L,)
        kernel: Filter of shape (K,)
    
    Returns:
        Output of shape (L - K + 1,)
    """
    K = kernel.shape[0]
    # x.unfold creates sliding windows: (L - K + 1, K)
    windows = x.unfold(0, K, 1)  # (out_len, K)
    return (windows * kernel).sum(dim=1)  # (out_len,)


# Test both implementations
x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0])
kernel = torch.tensor([1.0, 0.0, -1.0])

out_loop = conv1d_loop(x, kernel)
out_vec = conv1d_vectorized(x, kernel)

# Compare with PyTorch's F.conv1d (expects (B, C, L) input and (C_out, C_in, K) kernel)
out_pytorch = F.conv1d(
    x.view(1, 1, -1),        # (1, 1, 7)
    kernel.view(1, 1, -1)    # (1, 1, 3)
).squeeze()                   # (5,)

print(f"Loop output:      {out_loop}")
print(f"Vectorized output: {out_vec}")
print(f"PyTorch output:    {out_pytorch}")
assert torch.allclose(out_loop, out_vec)
assert torch.allclose(out_loop, out_pytorch)
print("All three match.")

### Worked Example: 2D Convolution with Unfolding

In [ ]:
def conv2d_naive(
    x: torch.Tensor,
    kernel: torch.Tensor,
) -> torch.Tensor:
    """2D convolution (correlation) for a single-channel input using unfold.
    
    Args:
        x: Input of shape (H, W)
        kernel: Filter of shape (K_h, K_w)
    
    Returns:
        Output of shape (H - K_h + 1, W - K_w + 1)
    """
    H, W = x.shape
    K_h, K_w = kernel.shape
    
    # Use unfold to extract all sliding windows
    # First unfold along height, then width
    patches = x.unfold(0, K_h, 1).unfold(1, K_w, 1)  # (H_out, W_out, K_h, K_w)
    return (patches * kernel).sum(dim=(-2, -1))  # (H_out, W_out)


# Test against F.conv2d
torch.manual_seed(42)
x_2d = torch.randn(6, 6)  # (H, W)
k_2d = torch.randn(3, 3)  # (K_h, K_w)

out_ours = conv2d_naive(x_2d, k_2d)  # (4, 4)
out_pt = F.conv2d(
    x_2d.view(1, 1, 6, 6),  # (B, C, H, W)
    k_2d.view(1, 1, 3, 3)   # (C_out, C_in, K_h, K_w)
).squeeze()                  # (4, 4)

print(f"Our output shape: {out_ours.shape}")
print(f"Max error: {(out_ours - out_pt).abs().max():.2e}")
assert torch.allclose(out_ours, out_pt, atol=1e-5)
print("2D convolution matches F.conv2d.")

### Exercise 2.1: Multi-Channel 2D Conv from Scratch + Edge Detection

1. Implement `conv2d_multichannel` that handles `(B, C_in, H, W)` inputs with `(C_out, C_in, K_h, K_w)` kernels.
2. Apply **Sobel** (horizontal and vertical edge) and **Laplacian** (all-direction edge) filters to a synthetic image.

In [ ]:
# EXERCISE: Implement multi-channel 2D convolution

def conv2d_multichannel(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Multi-channel 2D convolution from scratch.
    
    Args:
        x: Input tensor of shape (B, C_in, H, W)
        weight: Kernel tensor of shape (C_out, C_in, K_h, K_w)
        bias: Optional bias of shape (C_out,)
    
    Returns:
        Output tensor of shape (B, C_out, H_out, W_out)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Test your implementation
torch.manual_seed(42)
x_test = torch.randn(2, 3, 8, 8)  # (B, C_in, H, W)
w_test = torch.randn(4, 3, 3, 3)  # (C_out, C_in, K_h, K_w)
b_test = torch.randn(4)            # (C_out,)

# out_custom = conv2d_multichannel(x_test, w_test, b_test)
# out_ref = F.conv2d(x_test, w_test, b_test)
# assert torch.allclose(out_custom, out_ref, atol=1e-4), f"Max error: {(out_custom - out_ref).abs().max()}"
# print(f"Output shape: {out_custom.shape}")  # (2, 4, 6, 6)
# print("Multi-channel conv matches F.conv2d.")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def conv2d_multichannel(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Multi-channel 2D convolution from scratch using unfold.
    
    Args:
        x: Input tensor of shape (B, C_in, H, W)
        weight: Kernel tensor of shape (C_out, C_in, K_h, K_w)
        bias: Optional bias of shape (C_out,)
    
    Returns:
        Output tensor of shape (B, C_out, H_out, W_out)
    """
    B, C_in, H, W = x.shape  # (B, C_in, H, W)
    C_out, C_in_k, K_h, K_w = weight.shape  # (C_out, C_in, K_h, K_w)
    assert C_in == C_in_k, f"Channel mismatch: input has {C_in}, kernel has {C_in_k}"
    
    H_out = H - K_h + 1
    W_out = W - K_w + 1
    
    # Unfold input into column matrix: (B, C_in * K_h * K_w, H_out * W_out)
    x_unfold = F.unfold(x, kernel_size=(K_h, K_w))  # (B, C_in*K_h*K_w, L)
    
    # Reshape weight to (C_out, C_in * K_h * K_w)
    w_flat = weight.view(C_out, -1)  # (C_out, C_in*K_h*K_w)
    
    # Matrix multiply: (C_out, C_in*K_h*K_w) @ (B, C_in*K_h*K_w, L) -> (B, C_out, L)
    out = torch.einsum("oi,bil->bol", w_flat, x_unfold)  # (B, C_out, H_out*W_out)
    
    if bias is not None:
        out = out + bias.view(1, -1, 1)  # (B, C_out, H_out*W_out)
    
    return out.view(B, C_out, H_out, W_out)  # (B, C_out, H_out, W_out)


# Test implementation
torch.manual_seed(42)
x_test = torch.randn(2, 3, 8, 8)  # (B, C_in, H, W)
w_test = torch.randn(4, 3, 3, 3)  # (C_out, C_in, K_h, K_w)
b_test = torch.randn(4)            # (C_out,)

out_custom = conv2d_multichannel(x_test, w_test, b_test)  # (2, 4, 6, 6)
out_ref = F.conv2d(x_test, w_test, b_test)                # (2, 4, 6, 6)
assert torch.allclose(out_custom, out_ref, atol=1e-4), f"Max error: {(out_custom - out_ref).abs().max()}"
print(f"Output shape: {out_custom.shape}")  # (2, 4, 6, 6)
print("Multi-channel conv matches F.conv2d.")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# Create a synthetic test image: white square on black background
img = torch.zeros(1, 1, 64, 64)  # (B, C, H, W)
img[0, 0, 16:48, 16:48] = 1.0    # White square

# Define edge detection kernels
sobel_x = torch.tensor([[-1., 0., 1.],
                         [-2., 0., 2.],
                         [-1., 0., 1.]]).view(1, 1, 3, 3)  # (1, 1, 3, 3)

sobel_y = torch.tensor([[-1., -2., -1.],
                         [ 0.,  0.,  0.],
                         [ 1.,  2.,  1.]]).view(1, 1, 3, 3)  # (1, 1, 3, 3)

laplacian = torch.tensor([[ 0., 1., 0.],
                           [ 1., -4., 1.],
                           [ 0., 1., 0.]]).view(1, 1, 3, 3)  # (1, 1, 3, 3)

# Apply with our custom implementation
edges_x = conv2d_multichannel(img, sobel_x).squeeze()    # (62, 62)
edges_y = conv2d_multichannel(img, sobel_y).squeeze()    # (62, 62)
edges_lap = conv2d_multichannel(img, laplacian).squeeze() # (62, 62)
edges_mag = torch.sqrt(edges_x**2 + edges_y**2)          # (62, 62) -- Sobel magnitude

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
titles = ["Original", "Sobel X (vertical)", "Sobel Y (horizontal)", "Sobel magnitude", "Laplacian"]
images = [img.squeeze(), edges_x, edges_y, edges_mag, edges_lap]

for ax, title, im in zip(axes, titles, images):
    ax.imshow(im.detach().numpy(), cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Edge Detection with Hand-Crafted Convolution Kernels", fontsize=13)
plt.tight_layout()
plt.show()

---
## 2.2 -- Padding, Stride, Dilation: The Output Size Formula

The single most important formula for convolution arithmetic:

$$H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2 \cdot \text{padding} - \text{dilation} \cdot (\text{kernel} - 1) - 1}{\text{stride}} + 1 \right\rfloor$$

| Parameter | Effect | Typical Use |
|-----------|--------|------------|
| **Padding** | Adds border values (zero, reflect, replicate) | Preserve spatial dims ("same" convolution) |
| **Stride** | Skip positions when sliding kernel | Downsample (stride=2 halves resolution) |
| **Dilation** | Gaps between kernel elements | Expand receptive field without more params (atrous conv) |

**Padding modes:**
- **Zero padding:** Simple, most common. Adds zeros at borders.
- **Reflect padding:** Mirrors values at boundary. Better for textures (no sharp zero boundary).
- **Replicate padding:** Repeats edge values. Good for avoiding border artifacts.

### Worked Example: Output Size Calculations

In [ ]:
def output_size(
    input_size: int,
    kernel_size: int,
    padding: int = 0,
    stride: int = 1,
    dilation: int = 1,
) -> int:
    """Compute the output spatial dimension for a convolution."""
    return (input_size + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1


# Demonstrate various configurations
configs = [
    {"input_size": 32, "kernel_size": 3, "padding": 0, "stride": 1, "dilation": 1},
    {"input_size": 32, "kernel_size": 3, "padding": 1, "stride": 1, "dilation": 1},  # "same"
    {"input_size": 32, "kernel_size": 3, "padding": 1, "stride": 2, "dilation": 1},  # halve
    {"input_size": 32, "kernel_size": 3, "padding": 2, "stride": 1, "dilation": 2},  # dilated
    {"input_size": 32, "kernel_size": 5, "padding": 2, "stride": 1, "dilation": 1},  # 5x5 same
]

print(f"{'Input':>6} {'Kernel':>7} {'Pad':>4} {'Stride':>7} {'Dilation':>9} {'Output':>7}")
print("-" * 48)
for c in configs:
    out = output_size(**c)
    print(f"{c['input_size']:>6} {c['kernel_size']:>7} {c['padding']:>4} "
          f"{c['stride']:>7} {c['dilation']:>9} {out:>7}")

In [ ]:
# Demonstrate stride=2 halving -- the standard U-Net encoder downsample
torch.manual_seed(42)
x = torch.randn(1, 16, 64, 64)  # (B, C, H, W)

# Stride-2 conv: halves spatial dims, doubles channels (common pattern)
conv_down = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
out = conv_down(x)  # (1, 32, 32, 32)
print(f"Input:  {x.shape}  ->  Output: {out.shape}")
print(f"Spatial: 64 -> {out.shape[-1]}  (halved)")
print(f"Channels: 16 -> {out.shape[1]} (doubled)")
print("This is exactly what happens at each U-Net encoder stage.")

### Exercise 2.2: Same Padding and Output Size Computation

In [ ]:
# EXERCISE: Implement "same" padding computation and verify it

def compute_same_padding(
    kernel_size: int,
    stride: int = 1,
    dilation: int = 1,
) -> int:
    """Compute padding needed so output_size = ceil(input_size / stride).
    
    For stride=1 this means output_size == input_size.
    For stride=2 this means output_size == input_size // 2.
    
    Args:
        kernel_size: Size of the convolution kernel.
        stride: Convolution stride.
        dilation: Convolution dilation.
    
    Returns:
        Padding value (assumes symmetric padding).
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Test: for kernel=3, stride=1, dilation=1 -> padding should be 1
# Test: for kernel=5, stride=1, dilation=1 -> padding should be 2
# Test: for kernel=3, stride=1, dilation=2 -> padding should be 2

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def compute_same_padding(
    kernel_size: int,
    stride: int = 1,
    dilation: int = 1,
) -> int:
    """Compute padding needed so output_size = ceil(input_size / stride).
    
    For stride=1 this means output_size == input_size.
    For stride=2 this means output_size == input_size // 2.
    
    Args:
        kernel_size: Size of the convolution kernel.
        stride: Convolution stride.
        dilation: Convolution dilation.
    
    Returns:
        Padding value (assumes symmetric padding).
    """
    # Effective kernel size accounting for dilation
    effective_k = dilation * (kernel_size - 1) + 1
    # For "same" output: padding = (effective_k - 1) // 2
    # This works when stride=1. For stride>1, same formula gives
    # output = ceil(input / stride), which is the standard "same" definition.
    return (effective_k - 1) // 2


# Verify
test_cases = [
    {"kernel_size": 3, "stride": 1, "dilation": 1, "expected": 1},
    {"kernel_size": 5, "stride": 1, "dilation": 1, "expected": 2},
    {"kernel_size": 3, "stride": 1, "dilation": 2, "expected": 2},
    {"kernel_size": 3, "stride": 2, "dilation": 1, "expected": 1},
    {"kernel_size": 7, "stride": 1, "dilation": 1, "expected": 3},
]

for tc in test_cases:
    expected = tc.pop("expected")
    result = compute_same_padding(**tc)
    # Verify with actual convolution
    inp = torch.randn(1, 1, 32, 32)
    out = F.conv2d(inp, torch.randn(1, 1, tc["kernel_size"], tc["kernel_size"]),
                   padding=result, stride=tc["stride"], dilation=tc["dilation"])
    expected_spatial = (32 + tc["stride"] - 1) // tc["stride"]  # ceil(32/stride)
    assert result == expected, f"Expected padding={expected}, got {result} for {tc}"
    assert out.shape[-1] == expected_spatial, f"Expected output {expected_spatial}, got {out.shape[-1]}"
    print(f"kernel={tc['kernel_size']}, stride={tc['stride']}, dilation={tc['dilation']} "
          f"-> padding={result}, output_size={out.shape[-1]}")

print("All same-padding tests pass.")

---
## 2.3 -- Transposed Convolutions (Upsampling)

### Why we need upsampling

The U-Net decoder must increase spatial resolution to match the encoder. Transposed convolutions are one approach; bilinear interpolation + regular conv is another.

### What a transposed convolution actually does

A transposed convolution is **not** the inverse of convolution. It is the **gradient** (transpose of the Jacobian) of a convolution. Mechanically, it inserts zeros between input elements, then applies a regular convolution. The output size formula:

$$H_{\text{out}} = (H_{\text{in}} - 1) \cdot \text{stride} - 2 \cdot \text{padding} + \text{dilation} \cdot (\text{kernel} - 1) + \text{output\_padding} + 1$$

### Checkerboard artifacts

When `kernel_size` is not divisible by `stride`, different output positions receive contributions from different numbers of input elements. This causes a grid-like pattern called **checkerboard artifacts**. The preferred alternative in modern architectures (including many diffusion models) is `F.interpolate` (nearest or bilinear) followed by a regular convolution.

### Worked Example: Transposed Conv, Checkerboard, and Alternatives

In [ ]:
# Demonstrate the zero-insertion mechanism of transposed conv
torch.manual_seed(42)

x_small = torch.tensor([[1., 2.],
                         [3., 4.]]).view(1, 1, 2, 2)  # (1, 1, 2, 2)

# Transposed conv with stride=2: upsamples 2x2 -> 4x4
# Internally: insert zeros to get 3x3, then convolve with 3x3 kernel -> 4x4 (with padding=1 adjustment)
tconv = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False)

# Set identity-like kernel for visualization
with torch.no_grad():
    tconv.weight.fill_(0)
    tconv.weight[0, 0, 1, 1] = 1.0  # Center element only

out_tconv = tconv(x_small)  # (1, 1, 4, 4)
print("Input (2x2):")
print(x_small.squeeze())
print(f"\nTransposed conv output (4x4) with identity-center kernel:")
print(out_tconv.squeeze())
print(f"\nInput shape: {x_small.shape} -> Output shape: {out_tconv.shape}")

In [ ]:
# Demonstrate checkerboard artifacts
torch.manual_seed(42)

# Create a small feature map and upsample with a transposed conv
# that has kernel_size not divisible by stride (classic artifact trigger)
x_feat = torch.randn(1, 1, 8, 8)  # (1, 1, 8, 8)

# Bad: kernel_size=3, stride=2 -> checkerboard
tconv_bad = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, padding=1, output_padding=1)

# Good: kernel_size=4, stride=2 (divisible) -> less artifact
tconv_good = nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1)

# Alternative: interpolate + conv (preferred in practice)
conv_after_interp = nn.Conv2d(1, 1, kernel_size=3, padding=1)

out_bad = tconv_bad(x_feat)  # (1, 1, 16, 16)
out_good = tconv_good(x_feat)  # (1, 1, 16, 16)
out_interp = conv_after_interp(
    F.interpolate(x_feat, scale_factor=2, mode="nearest")  # (1, 1, 16, 16)
)  # (1, 1, 16, 16)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ["Input (8x8)", "TransConv k=3,s=2\n(checkerboard)",
          "TransConv k=4,s=2\n(less artifact)", "Interpolate + Conv\n(preferred)"]
images = [x_feat.squeeze(), out_bad.squeeze(), out_good.squeeze(), out_interp.squeeze()]

for ax, title, im in zip(axes, titles, images):
    ax.imshow(im.detach().numpy(), cmap="viridis")
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Transposed Convolution: Checkerboard Artifacts", fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 2.3: Build Upsampling Blocks Both Ways

In [ ]:
# EXERCISE: Implement two upsampling blocks and compare

class UpsampleTransConv(nn.Module):
    """Upsample 2x using transposed convolution."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # YOUR CODE HERE: transposed conv that doubles spatial dims
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


class UpsampleInterpConv(nn.Module):
    """Upsample 2x using nearest interpolation + 3x3 conv."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # YOUR CODE HERE: interpolation followed by regular conv
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


# Test that both produce the correct output shape
# x_test = torch.randn(2, 32, 16, 16)
# up1 = UpsampleTransConv(32, 16)
# up2 = UpsampleInterpConv(32, 16)
# print(f"TransConv upsample: {x_test.shape} -> {up1(x_test).shape}")
# print(f"Interp+Conv upsample: {x_test.shape} -> {up2(x_test).shape}")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class UpsampleTransConv(nn.Module):
    """Upsample 2x using transposed convolution."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # kernel=4, stride=2, padding=1 -> doubles spatial dims cleanly
        self.tconv = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=4, stride=2, padding=1
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.tconv(x)  # (B, C_out, 2*H, 2*W)


class UpsampleInterpConv(nn.Module):
    """Upsample 2x using nearest interpolation + 3x3 conv."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2, mode="nearest")  # (B, C, 2*H, 2*W)
        return self.conv(x)  # (B, C_out, 2*H, 2*W)


# Test both
torch.manual_seed(42)
x_test = torch.randn(2, 32, 16, 16)  # (B, C, H, W)
up1 = UpsampleTransConv(32, 16)
up2 = UpsampleInterpConv(32, 16)

out1 = up1(x_test)  # (2, 16, 32, 32)
out2 = up2(x_test)  # (2, 16, 32, 32)

print(f"TransConv upsample:   {x_test.shape} -> {out1.shape}")
print(f"Interp+Conv upsample: {x_test.shape} -> {out2.shape}")

# Compare parameter counts
p1 = sum(p.numel() for p in up1.parameters())
p2 = sum(p.numel() for p in up2.parameters())
print(f"\nTransConv params:   {p1:,}")
print(f"Interp+Conv params: {p2:,}")
print(f"\nInterp+Conv is preferred in many diffusion models (no checkerboard).")

---
## 2.4 -- Depthwise Separable Convolutions

### Computational cost breakdown

| Convolution Type | Parameters | MACs (for H x W output) |
|-----------------|------------|------------------------|
| **Standard** 3x3 | C_in * C_out * 9 | C_in * C_out * 9 * H * W |
| **Depthwise** 3x3 | C_in * 9 | C_in * 9 * H * W |
| **Pointwise** 1x1 | C_in * C_out | C_in * C_out * H * W |
| **Separable** (DW+PW) | C_in * 9 + C_in * C_out | (C_in * 9 + C_in * C_out) * H * W |

The ratio of separable to standard parameters: `(9 + C_out) / (9 * C_out)`. For C_out=64 and 3x3 kernel, this is approximately **1/9** of the parameters.

**Depthwise:** Each input channel is convolved independently (groups=C_in).
**Pointwise:** A 1x1 conv mixes information across channels.

### Worked Example: Depthwise Separable vs Standard Conv

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """Depthwise separable convolution = depthwise + pointwise."""
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, padding: int = 1):
        super().__init__()
        # Depthwise: each channel convolved independently (groups=in_channels)
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            padding=padding, groups=in_channels, bias=False
        )
        # Pointwise: 1x1 conv to mix channels
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)   # (B, C_in, H, W)
        x = self.pointwise(x)   # (B, C_out, H, W)
        return x


# Compare parameter counts
C_in, C_out = 64, 128
standard = nn.Conv2d(C_in, C_out, kernel_size=3, padding=1)
separable = DepthwiseSeparableConv(C_in, C_out)

params_std = sum(p.numel() for p in standard.parameters())
params_sep = sum(p.numel() for p in separable.parameters())

print(f"Standard conv params:  {params_std:>8,}")
print(f"Separable conv params: {params_sep:>8,}")
print(f"Ratio: {params_sep / params_std:.3f} ({params_std / params_sep:.1f}x fewer)")

# Verify same output shape
torch.manual_seed(42)
x_test = torch.randn(2, C_in, 16, 16)  # (B, C_in, H, W)
print(f"\nStandard output shape:  {standard(x_test).shape}")   # (2, 128, 16, 16)
print(f"Separable output shape: {separable(x_test).shape}")    # (2, 128, 16, 16)

### Exercise 2.4: Replace Standard with Separable, Compare Speed

In [ ]:
# EXERCISE: Build two small networks -- one with standard convs, one with separable.
# Compare parameter count and forward pass speed.

class SmallConvNet(nn.Module):
    """3-layer ConvNet using standard convolutions."""
    def __init__(self, use_separable: bool = False):
        super().__init__()
        # YOUR CODE HERE
        # Build 3 conv layers: 3->32->64->128, each with kernel=3, padding=1
        # If use_separable, use DepthwiseSeparableConv instead of nn.Conv2d
        # Add ReLU activations between layers
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class SmallConvNet(nn.Module):
    """3-layer ConvNet with standard or separable convolutions."""
    def __init__(self, use_separable: bool = False):
        super().__init__()
        ConvBlock = DepthwiseSeparableConv if use_separable else lambda c_in, c_out, **kw: nn.Conv2d(c_in, c_out, kernel_size=3, padding=1)
        
        self.layers = nn.Sequential(
            ConvBlock(3, 32),
            nn.ReLU(),
            ConvBlock(32, 64),
            nn.ReLU(),
            ConvBlock(64, 128),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)  # (B, 128, H, W)


net_std = SmallConvNet(use_separable=False)
net_sep = SmallConvNet(use_separable=True)

params_std = sum(p.numel() for p in net_std.parameters())
params_sep = sum(p.numel() for p in net_sep.parameters())
print(f"Standard params:  {params_std:,}")
print(f"Separable params: {params_sep:,}")
print(f"Reduction: {params_sep / params_std:.3f}x")

# Speed comparison
x_bench = torch.randn(8, 3, 64, 64)

# Warm up
for _ in range(3):
    _ = net_std(x_bench)
    _ = net_sep(x_bench)

n_trials = 20
t0 = time.time()
for _ in range(n_trials):
    _ = net_std(x_bench)
t_std = (time.time() - t0) / n_trials

t0 = time.time()
for _ in range(n_trials):
    _ = net_sep(x_bench)
t_sep = (time.time() - t0) / n_trials

print(f"\nStandard forward:  {t_std*1000:.2f} ms")
print(f"Separable forward: {t_sep*1000:.2f} ms")
print(f"Speedup: {t_std/t_sep:.2f}x")

---
## 2.5 -- Batch Normalization

### Motivation

The original motivation was "internal covariate shift" -- the distribution of layer inputs changes during training as preceding layers update. While this explanation is [debated](https://arxiv.org/abs/1805.11604), the practical benefits are clear: BatchNorm enables higher learning rates and faster convergence.

### Mechanics

For a feature map `x` of shape `(B, C, H, W)`, BatchNorm computes per-channel statistics across the batch and spatial dimensions:

1. Compute mean and variance over `(B, H, W)` for each channel
2. Normalize: `x_hat = (x - mean) / sqrt(var + eps)`
3. Scale and shift: `y = gamma * x_hat + beta` (learnable per channel)

### Train vs eval mode

| Mode | Statistics used | Running stats |
|------|---------------|---------------|
| **Training** | Batch statistics (mean/var of current batch) | Updated via EMA: `running = momentum * running + (1 - momentum) * batch` |
| **Eval** | Running statistics (accumulated during training) | Frozen |

### Worked Example: BatchNorm from Scratch

In [ ]:
class BatchNorm2dFromScratch(nn.Module):
    """Batch normalization for (B, C, H, W) tensors, implemented from scratch."""
    
    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        # Learnable affine parameters
        self.gamma = nn.Parameter(torch.ones(num_features))   # (C,)
        self.beta = nn.Parameter(torch.zeros(num_features))   # (C,)
        
        # Running statistics (not parameters -- not updated by optimizer)
        self.register_buffer("running_mean", torch.zeros(num_features))  # (C,)
        self.register_buffer("running_var", torch.ones(num_features))    # (C,)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.
        
        Args:
            x: Input of shape (B, C, H, W)
        Returns:
            Normalized output of shape (B, C, H, W)
        """
        if self.training:
            # Compute batch statistics over (B, H, W) for each channel
            mean = x.mean(dim=(0, 2, 3))  # (C,)
            var = x.var(dim=(0, 2, 3), unbiased=False)  # (C,)
            
            # Update running stats (EMA)
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            mean = self.running_mean  # (C,)
            var = self.running_var    # (C,)
        
        # Normalize: reshape stats to (1, C, 1, 1) for broadcasting
        mean = mean.view(1, -1, 1, 1)   # (1, C, 1, 1)
        var = var.view(1, -1, 1, 1)      # (1, C, 1, 1)
        gamma = self.gamma.view(1, -1, 1, 1)  # (1, C, 1, 1)
        beta = self.beta.view(1, -1, 1, 1)    # (1, C, 1, 1)
        
        x_hat = (x - mean) / torch.sqrt(var + self.eps)  # (B, C, H, W)
        return gamma * x_hat + beta  # (B, C, H, W)


# Verify against PyTorch
torch.manual_seed(42)
x = torch.randn(4, 8, 16, 16)  # (B, C, H, W)

bn_ours = BatchNorm2dFromScratch(8)
bn_ref = nn.BatchNorm2d(8)

# Both in training mode
out_ours = bn_ours(x)  # (4, 8, 16, 16)
out_ref = bn_ref(x)    # (4, 8, 16, 16)

print(f"Output shape: {out_ours.shape}")
print(f"Max error: {(out_ours - out_ref).abs().max():.2e}")
assert torch.allclose(out_ours, out_ref, atol=1e-5)
print("BatchNorm from scratch matches nn.BatchNorm2d.")

# Verify running stats were updated
print(f"\nRunning mean (first 4 channels): {bn_ours.running_mean[:4].tolist()}")
print(f"Running var  (first 4 channels): {bn_ours.running_var[:4].tolist()}")

### Exercise 2.5: Train With and Without BatchNorm

In [ ]:
# EXERCISE: Build two small ConvNets (with/without BatchNorm), train on synthetic data,
# and compare convergence speed.

# Hint: Use a simple 3-layer ConvNet with global average pooling -> linear classifier.
# Create synthetic data with torch.randn for inputs and random labels.
# Train for ~200 steps, plot loss curves.

# YOUR CODE HERE
pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class ConvClassifier(nn.Module):
    """Simple ConvNet classifier with optional BatchNorm."""
    def __init__(self, use_batchnorm: bool = False, num_classes: int = 10):
        super().__init__()
        self.use_bn = use_batchnorm
        
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        
        if use_batchnorm:
            self.bn1 = nn.BatchNorm2d(32)
            self.bn2 = nn.BatchNorm2d(64)
            self.bn3 = nn.BatchNorm2d(128)
        
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)                                      # (B, 32, H, W)
        x = self.bn1(x) if self.use_bn else x                 # (B, 32, H, W)
        x = F.relu(x)                                          # (B, 32, H, W)
        x = F.max_pool2d(x, 2)                                 # (B, 32, H/2, W/2)
        
        x = self.conv2(x)                                      # (B, 64, H/2, W/2)
        x = self.bn2(x) if self.use_bn else x                 # (B, 64, H/2, W/2)
        x = F.relu(x)                                          # (B, 64, H/2, W/2)
        x = F.max_pool2d(x, 2)                                 # (B, 64, H/4, W/4)
        
        x = self.conv3(x)                                      # (B, 128, H/4, W/4)
        x = self.bn3(x) if self.use_bn else x                 # (B, 128, H/4, W/4)
        x = F.relu(x)                                          # (B, 128, H/4, W/4)
        
        x = x.mean(dim=(2, 3))                                 # (B, 128) -- global avg pool
        return self.fc(x)                                      # (B, num_classes)


# Train both variants on synthetic data
torch.manual_seed(42)
num_steps = 200
batch_size = 32

# Synthetic dataset: random images, random labels
X_train = torch.randn(256, 3, 16, 16)  # (N, C, H, W)
y_train = torch.randint(0, 10, (256,))  # (N,)

losses = {"Without BatchNorm": [], "With BatchNorm": []}

for use_bn, label in [(False, "Without BatchNorm"), (True, "With BatchNorm")]:
    torch.manual_seed(42)
    model = ConvClassifier(use_batchnorm=use_bn)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    for step in range(num_steps):
        idx = torch.randint(0, len(X_train), (batch_size,))
        xb, yb = X_train[idx], y_train[idx]
        
        logits = model(xb)  # (B, 10)
        loss = F.cross_entropy(logits, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses[label].append(loss.item())

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
for label, loss_list in losses.items():
    ax.plot(loss_list, label=label, alpha=0.8)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Convergence: BatchNorm vs No BatchNorm")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2.6 -- Group Normalization (Why Diffusion Models Prefer It)

### The problem with BatchNorm

BatchNorm computes statistics across the batch dimension. When the batch is small (common in diffusion training due to GPU memory) or 1 (always at inference), these statistics become noisy or undefined. This makes BatchNorm unsuitable for diffusion models.

### GroupNorm: batch-independent normalization

GroupNorm divides channels into **groups** and normalizes within each group, independently per sample. It never looks at other samples in the batch.

### Normalization zoo -- all the same operation with different grouping

| Normalization | Groups | Normalize over | Batch dependent? |
|--------------|--------|---------------|------------------|
| **BatchNorm** | -- | (B, H, W) per channel | Yes |
| **LayerNorm** | 1 group = all channels | (C, H, W) per sample | No |
| **InstanceNorm** | C groups = 1 channel each | (H, W) per channel per sample | No |
| **GroupNorm** | G groups (typically 32) | (C/G, H, W) per group per sample | No |

**Why diffusion uses GroupNorm:**
- Training uses small batches (large images eat GPU memory)
- Inference is always batch_size=1 (generating one image at a time)
- GroupNorm gives identical results regardless of batch size
- Typical config: 32 groups (as in the original DDPM U-Net)

### Worked Example: GroupNorm from Scratch

In [ ]:
class GroupNormFromScratch(nn.Module):
    """Group normalization for (B, C, H, W) tensors, implemented from scratch."""
    
    def __init__(self, num_groups: int, num_channels: int, eps: float = 1e-5):
        super().__init__()
        assert num_channels % num_groups == 0, "num_channels must be divisible by num_groups"
        self.num_groups = num_groups
        self.num_channels = num_channels
        self.eps = eps
        
        self.gamma = nn.Parameter(torch.ones(num_channels))   # (C,)
        self.beta = nn.Parameter(torch.zeros(num_channels))   # (C,)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.
        
        Args:
            x: Input of shape (B, C, H, W)
        Returns:
            Normalized output of shape (B, C, H, W)
        """
        B, C, H, W = x.shape  # (B, C, H, W)
        G = self.num_groups
        
        # Reshape to (B, G, C//G, H, W) -- group channels
        x = x.view(B, G, C // G, H, W)  # (B, G, C//G, H, W)
        
        # Compute mean and var within each group (over channels-in-group, H, W)
        mean = x.mean(dim=(2, 3, 4), keepdim=True)  # (B, G, 1, 1, 1)
        var = x.var(dim=(2, 3, 4), keepdim=True, unbiased=False)  # (B, G, 1, 1, 1)
        
        # Normalize
        x = (x - mean) / torch.sqrt(var + self.eps)  # (B, G, C//G, H, W)
        
        # Reshape back to (B, C, H, W)
        x = x.view(B, C, H, W)  # (B, C, H, W)
        
        # Apply learnable affine
        gamma = self.gamma.view(1, C, 1, 1)  # (1, C, 1, 1)
        beta = self.beta.view(1, C, 1, 1)    # (1, C, 1, 1)
        return gamma * x + beta  # (B, C, H, W)


# Verify against PyTorch
torch.manual_seed(42)
x = torch.randn(4, 32, 8, 8)  # (B, C, H, W)

gn_ours = GroupNormFromScratch(num_groups=8, num_channels=32)
gn_ref = nn.GroupNorm(num_groups=8, num_channels=32)

out_ours = gn_ours(x)  # (4, 32, 8, 8)
out_ref = gn_ref(x)    # (4, 32, 8, 8)

print(f"Output shape: {out_ours.shape}")
print(f"Max error: {(out_ours - out_ref).abs().max():.2e}")
assert torch.allclose(out_ours, out_ref, atol=1e-5)
print("GroupNorm from scratch matches nn.GroupNorm.")

In [ ]:
# Demonstrate batch-size independence (GroupNorm vs BatchNorm)
torch.manual_seed(42)
x_single = torch.randn(1, 32, 8, 8)  # (1, C, H, W) -- single sample
x_batch = torch.cat([x_single, torch.randn(3, 32, 8, 8)], dim=0)  # (4, C, H, W)

# GroupNorm: same output for x_single[0] regardless of batch
gn = nn.GroupNorm(8, 32)
gn_single = gn(x_single)[0]  # (32, 8, 8)
gn_batch = gn(x_batch)[0]    # (32, 8, 8)

# BatchNorm: DIFFERENT output depending on batch
bn = nn.BatchNorm2d(32)
bn.train()
bn_single = bn(x_single)[0]  # (32, 8, 8)
# Reset running stats to avoid contamination
bn = nn.BatchNorm2d(32)
bn.train()
bn_batch = bn(x_batch)[0]    # (32, 8, 8)

print("GroupNorm difference (same sample, different batch sizes):")
print(f"  Max diff: {(gn_single - gn_batch).abs().max():.2e}")
print("\nBatchNorm difference (same sample, different batch sizes):")
print(f"  Max diff: {(bn_single - bn_batch).abs().max():.4f}")
print("\nGroupNorm output is identical regardless of batch. BatchNorm is not.")
print("This is why diffusion models use GroupNorm.")

### Exercise 2.6: Implement All Four Normalizations

In [ ]:
# EXERCISE: Implement Batch/Layer/Instance/Group Norm as the SAME function
# with different "grouping" parameters.

# The key insight: all four normalizations compute mean/var over different axes.
# They can all be expressed as: reshape to (B, G, C//G, H, W), normalize over (2,3,4).
#   - BatchNorm:    special case (normalizes over batch, not groups)
#   - LayerNorm:    G = 1
#   - InstanceNorm: G = C
#   - GroupNorm:    G = num_groups

def unified_norm(
    x: torch.Tensor,
    norm_type: str,
    num_groups: int = 32,
    eps: float = 1e-5,
) -> torch.Tensor:
    """Unified normalization (without learnable parameters, for demonstration).
    
    Args:
        x: Input of shape (B, C, H, W)
        norm_type: One of 'batch', 'layer', 'instance', 'group'
        num_groups: Number of groups (only used when norm_type='group')
        eps: Epsilon for numerical stability
    
    Returns:
        Normalized tensor of shape (B, C, H, W)
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def unified_norm(
    x: torch.Tensor,
    norm_type: str,
    num_groups: int = 32,
    eps: float = 1e-5,
) -> torch.Tensor:
    """Unified normalization (without learnable parameters, for demonstration).
    
    Args:
        x: Input of shape (B, C, H, W)
        norm_type: One of 'batch', 'layer', 'instance', 'group'
        num_groups: Number of groups (only used when norm_type='group')
        eps: Epsilon for numerical stability
    
    Returns:
        Normalized tensor of shape (B, C, H, W)
    """
    B, C, H, W = x.shape  # (B, C, H, W)
    
    if norm_type == "batch":
        # Mean/var over (B, H, W) for each channel
        mean = x.mean(dim=(0, 2, 3), keepdim=True)  # (1, C, 1, 1)
        var = x.var(dim=(0, 2, 3), keepdim=True, unbiased=False)  # (1, C, 1, 1)
        return (x - mean) / torch.sqrt(var + eps)  # (B, C, H, W)
    
    # For layer/instance/group, reshape to (B, G, C//G, H, W)
    if norm_type == "layer":
        G = 1  # One group = all channels
    elif norm_type == "instance":
        G = C  # Each channel is its own group
    elif norm_type == "group":
        G = num_groups
    else:
        raise ValueError(f"Unknown norm_type: {norm_type}")
    
    x_grouped = x.view(B, G, C // G, H, W)  # (B, G, C//G, H, W)
    mean = x_grouped.mean(dim=(2, 3, 4), keepdim=True)  # (B, G, 1, 1, 1)
    var = x_grouped.var(dim=(2, 3, 4), keepdim=True, unbiased=False)  # (B, G, 1, 1, 1)
    x_normed = (x_grouped - mean) / torch.sqrt(var + eps)  # (B, G, C//G, H, W)
    return x_normed.view(B, C, H, W)  # (B, C, H, W)


# Verify each against PyTorch
torch.manual_seed(42)
x = torch.randn(4, 32, 8, 8)  # (B, C, H, W)

# LayerNorm: PyTorch normalizes over last N dims
ln = nn.LayerNorm([32, 8, 8], elementwise_affine=False)
out_ln_ref = ln(x)
out_ln_ours = unified_norm(x, "layer")
print(f"LayerNorm max error:    {(out_ln_ours - out_ln_ref).abs().max():.2e}")

# InstanceNorm
inst = nn.InstanceNorm2d(32, affine=False)
out_in_ref = inst(x)
out_in_ours = unified_norm(x, "instance")
print(f"InstanceNorm max error: {(out_in_ours - out_in_ref).abs().max():.2e}")

# GroupNorm
gn = nn.GroupNorm(8, 32, affine=False)
out_gn_ref = gn(x)
out_gn_ours = unified_norm(x, "group", num_groups=8)
print(f"GroupNorm max error:    {(out_gn_ours - out_gn_ref).abs().max():.2e}")

# BatchNorm (train mode, no affine)
bn = nn.BatchNorm2d(32, affine=False)
bn.train()
out_bn_ref = bn(x)
out_bn_ours = unified_norm(x, "batch")
print(f"BatchNorm max error:    {(out_bn_ours - out_bn_ref).abs().max():.2e}")

print("\nAll normalizations are the same operation with different grouping axes.")

---
## 2.7 -- Residual Connections: Gradient Flow Intuition

### The degradation problem

Without residual connections, deeper networks perform **worse** than shallower ones -- not due to overfitting, but because optimization becomes harder. Gradients vanish or explode through many layers of nonlinear transformations.

### The residual connection: y = F(x) + x

Instead of learning a mapping `H(x)`, learn the **residual** `F(x) = H(x) - x`. The output is `y = F(x) + x`. This has two key benefits:

1. **Gradient flow:** The identity shortcut provides a direct path for gradients: `dy/dx = dF/dx + I`. Even if `dF/dx` is small, the gradient is at least `I`.
2. **Easy identity:** If the optimal mapping is close to identity, the network only needs to push `F(x)` toward zero -- much easier than learning the identity explicitly.

### Dimension mismatch

When `x` and `F(x)` have different shapes (different channel count or spatial size), use a **1x1 projection convolution** on the shortcut: `y = F(x) + W_s * x`.

### Pre-activation vs post-activation

| Variant | Order | Used in |
|---------|-------|---------|
| **Post-activation** (original ResNet) | Conv -> BN -> ReLU -> Conv -> BN -> Add -> ReLU | ResNet v1 |
| **Pre-activation** | BN -> ReLU -> Conv -> BN -> ReLU -> Conv -> Add | ResNet v2, many diffusion U-Nets |

Diffusion models typically use a variant: Conv -> GroupNorm -> SiLU -> Conv -> GroupNorm + shortcut (with SiLU instead of ReLU).

*Reference: [Deep Residual Learning](https://arxiv.org/abs/1512.03385) -- He et al. 2015*

### Worked Example: ResBlock with Gradient Analysis

In [ ]:
class ResBlock(nn.Module):
    """Residual block: Conv -> GroupNorm -> SiLU -> Conv -> GroupNorm + shortcut.
    
    This is the building block of diffusion U-Nets.
    """
    def __init__(self, channels: int, num_groups: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups, channels)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x                          # (B, C, H, W) -- identity shortcut
        x = self.conv1(x)                     # (B, C, H, W)
        x = self.gn1(x)                       # (B, C, H, W)
        x = F.silu(x)                         # (B, C, H, W)
        x = self.conv2(x)                     # (B, C, H, W)
        x = self.gn2(x)                       # (B, C, H, W)
        return x + residual                   # (B, C, H, W)


class PlainBlock(nn.Module):
    """Same architecture but WITHOUT the residual connection."""
    def __init__(self, channels: int, num_groups: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups, channels)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)                     # (B, C, H, W)
        x = self.gn1(x)                       # (B, C, H, W)
        x = F.silu(x)                         # (B, C, H, W)
        x = self.conv2(x)                     # (B, C, H, W)
        x = self.gn2(x)                       # (B, C, H, W)
        return x                              # (B, C, H, W) -- no shortcut!


# Compare gradient magnitudes through deep stacks
torch.manual_seed(42)
depth = 20
channels = 64  # Must be divisible by 32 (num_groups)

res_blocks = nn.Sequential(*[ResBlock(channels) for _ in range(depth)])
plain_blocks = nn.Sequential(*[PlainBlock(channels) for _ in range(depth)])

x = torch.randn(1, channels, 16, 16, requires_grad=True)  # (1, 64, 16, 16)

# Forward + backward through residual network
out_res = res_blocks(x)  # (1, 64, 16, 16)
loss_res = out_res.sum()
loss_res.backward()
grad_res = x.grad.norm().item()

x.grad = None  # Reset gradient

# Forward + backward through plain network
out_plain = plain_blocks(x)  # (1, 64, 16, 16)
loss_plain = out_plain.sum()
loss_plain.backward()
grad_plain = x.grad.norm().item()

print(f"Depth: {depth} blocks ({depth * 2} conv layers)")
print(f"Gradient norm at input (residual):  {grad_res:.4f}")
print(f"Gradient norm at input (plain):     {grad_plain:.6f}")
print(f"Ratio (residual / plain): {grad_res / (grad_plain + 1e-12):.1f}x stronger")
print("\nResidual connections preserve gradient flow through deep networks.")

### Exercise 2.7: ResBlock with Optional Dimension Change (The Diffusion U-Net Block)

In [ ]:
# EXERCISE: Build a ResBlock that handles channel dimension changes.
# When in_channels != out_channels, the shortcut needs a 1x1 projection.
# This IS the building block used in diffusion U-Nets.

class ResBlockWithProjection(nn.Module):
    """Residual block with optional 1x1 projection for channel changes.
    
    Architecture:
        Conv 3x3 -> GroupNorm -> SiLU -> Conv 3x3 -> GroupNorm + shortcut
    
    If in_channels != out_channels, the shortcut uses a 1x1 conv.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_groups: int = 32,
    ):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        raise NotImplementedError


# Test cases:
# 1. Same channels (identity shortcut)
# 2. Different channels (1x1 projection shortcut)
# x1 = torch.randn(2, 64, 16, 16)
# block_same = ResBlockWithProjection(64, 64)
# print(f"Same channels: {x1.shape} -> {block_same(x1).shape}")

# block_diff = ResBlockWithProjection(64, 128)
# print(f"Diff channels: {x1.shape} -> {block_diff(x1).shape}")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class ResBlockWithProjection(nn.Module):
    """Residual block with optional 1x1 projection for channel changes.
    
    Architecture:
        Conv 3x3 -> GroupNorm -> SiLU -> Conv 3x3 -> GroupNorm + shortcut
    
    If in_channels != out_channels, the shortcut uses a 1x1 conv.
    This is the exact block used in diffusion U-Nets.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_groups: int = 32,
    ):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        
        # 1x1 projection shortcut if channels change
        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)           # (B, C_out, H, W)
        x = self.conv1(x)                     # (B, C_out, H, W)
        x = self.gn1(x)                       # (B, C_out, H, W)
        x = F.silu(x)                         # (B, C_out, H, W)
        x = self.conv2(x)                     # (B, C_out, H, W)
        x = self.gn2(x)                       # (B, C_out, H, W)
        return x + residual                   # (B, C_out, H, W)


# Test both cases
torch.manual_seed(42)
x1 = torch.randn(2, 64, 16, 16)  # (B, C, H, W)

# Same channels -- identity shortcut
block_same = ResBlockWithProjection(64, 64)
out_same = block_same(x1)  # (2, 64, 16, 16)
print(f"Same channels: {x1.shape} -> {out_same.shape}")
print(f"  Shortcut type: {type(block_same.shortcut).__name__}")

# Different channels -- 1x1 projection
block_diff = ResBlockWithProjection(64, 128)
out_diff = block_diff(x1)  # (2, 128, 16, 16)
print(f"Diff channels: {x1.shape} -> {out_diff.shape}")
print(f"  Shortcut type: {type(block_diff.shortcut).__name__}")

# Count parameters
params_same = sum(p.numel() for p in block_same.parameters())
params_diff = sum(p.numel() for p in block_diff.parameters())
print(f"\nParams (same channels):  {params_same:,}")
print(f"Params (diff channels):  {params_diff:,}")
print(f"Extra from 1x1 projection: {params_diff - params_same:,}")

---
## Capstone Exercise: CIFAR-10 ConvNet with Custom Building Blocks

Build a small ConvNet classifier from the components developed in this module:

- **Conv2d layers** for feature extraction
- **GroupNorm** (not BatchNorm -- as used in diffusion models)
- **Residual connections** with 1x1 projection when channels change
- **SiLU activation** (not ReLU -- diffusion models use SiLU)
- **Stride-2 convolution** for downsampling (not pooling)

Target: **>70% accuracy** on CIFAR-10 test set.

This architecture is a simplified version of the U-Net encoder used in diffusion models.

In [ ]:
# EXERCISE: Build and train the CIFAR-10 ConvNet

# Architecture suggestion:
#   Input (3, 32, 32)
#   -> Conv2d(3, 64, 3, padding=1)   -> (64, 32, 32)
#   -> ResBlock(64, 64)              -> (64, 32, 32)
#   -> Conv2d(64, 128, 3, stride=2, padding=1)  -> (128, 16, 16)   # Downsample
#   -> ResBlock(128, 128)            -> (128, 16, 16)
#   -> Conv2d(128, 256, 3, stride=2, padding=1) -> (256, 8, 8)     # Downsample
#   -> ResBlock(256, 256)            -> (256, 8, 8)
#   -> Global Average Pool           -> (256,)
#   -> Linear(256, 10)               -> (10,)

# YOUR CODE HERE
pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

import torchvision
import torchvision.transforms as transforms


class CapstoneCIFAR10Net(nn.Module):
    """Small ConvNet for CIFAR-10 using diffusion-model building blocks.
    
    Uses GroupNorm, SiLU, residual connections, and stride-2 downsampling --
    the same components found in diffusion U-Net encoders.
    """
    def __init__(self, num_classes: int = 10):
        super().__init__()
        
        # Initial projection
        self.stem = nn.Conv2d(3, 64, 3, padding=1)                # (B, 64, 32, 32)
        
        # Stage 1: 32x32 resolution
        self.res1 = ResBlockWithProjection(64, 64, num_groups=32)  # (B, 64, 32, 32)
        
        # Downsample 32x32 -> 16x16
        self.down1 = nn.Conv2d(64, 128, 3, stride=2, padding=1)   # (B, 128, 16, 16)
        
        # Stage 2: 16x16 resolution
        self.res2 = ResBlockWithProjection(128, 128, num_groups=32)  # (B, 128, 16, 16)
        
        # Downsample 16x16 -> 8x8
        self.down2 = nn.Conv2d(128, 256, 3, stride=2, padding=1)  # (B, 256, 8, 8)
        
        # Stage 3: 8x8 resolution
        self.res3 = ResBlockWithProjection(256, 256, num_groups=32)  # (B, 256, 8, 8)
        
        # Classification head
        self.norm_out = nn.GroupNorm(32, 256)
        self.fc = nn.Linear(256, num_classes)                     # (B, 10)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self.stem(x))      # (B, 64, 32, 32)
        x = self.res1(x)              # (B, 64, 32, 32)
        x = F.silu(self.down1(x))     # (B, 128, 16, 16)
        x = self.res2(x)              # (B, 128, 16, 16)
        x = F.silu(self.down2(x))     # (B, 256, 8, 8)
        x = self.res3(x)              # (B, 256, 8, 8)
        x = F.silu(self.norm_out(x))  # (B, 256, 8, 8)
        x = x.mean(dim=(2, 3))        # (B, 256) -- global average pool
        return self.fc(x)             # (B, 10)


model = CapstoneCIFAR10Net().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# Verify output shape
test_input = torch.randn(2, 3, 32, 32).to(device)
test_output = model(test_input)  # (2, 10)
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}")

> **macOS note:** If you encounter multiprocessing errors, change `num_workers=2` to `num_workers=0` in the DataLoader calls below. macOS sometimes has issues with forked worker processes.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# Data augmentation (standard for CIFAR-10)
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

print(f"Training samples: {len(trainset):,}")
print(f"Test samples:     {len(testset):,}")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

torch.manual_seed(42)
model = CapstoneCIFAR10Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

num_epochs = 15
train_losses = []
test_accs = []

for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)  # (B, 3, 32, 32), (B,)
        
        logits = model(images)  # (B, 10)
        loss = F.cross_entropy(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    scheduler.step()
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    
    # --- Evaluation ---
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)  # (B, 10)
            preds = logits.argmax(dim=1)  # (B,)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    acc = 100.0 * correct / total
    test_accs.append(acc)
    print(f"Epoch {epoch+1:2d}/{num_epochs} | Loss: {avg_loss:.4f} | Test Acc: {acc:.1f}%")

print(f"\nFinal test accuracy: {test_accs[-1]:.1f}%")
if test_accs[-1] >= 70.0:
    print("Target of >70% accuracy achieved.")
else:
    print("Note: More epochs or tuning needed to reach 70%.")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, num_epochs + 1), train_losses, "b-o", markersize=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, num_epochs + 1), test_accs, "r-o", markersize=4)
ax2.axhline(y=70, color="gray", linestyle="--", label="70% target")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Test Accuracy (%)")
ax2.set_title("Test Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("CIFAR-10 ConvNet: GroupNorm + Residual + SiLU (Diffusion-Style Encoder)", fontsize=12)
plt.tight_layout()
plt.show()

---
## Summary

This module covered the core convolutional building blocks used in diffusion U-Nets:

| Component | Key Takeaway | Diffusion Relevance |
|-----------|-------------|--------------------|
| **Convolution** | Sliding kernel, parameter sharing, translation equivariance | Every layer in the U-Net |
| **Padding/Stride/Dilation** | Output size formula; stride-2 for downsampling | Encoder downsampling |
| **Transposed Conv** | Upsampling via zero-insertion; prefer interpolate+conv | Decoder upsampling |
| **Depthwise Separable** | ~9x fewer params for 3x3; groups=C_in | Some efficient U-Net variants |
| **BatchNorm** | Per-channel normalization across batch | NOT used in diffusion (batch-dependent) |
| **GroupNorm** | Per-group normalization, batch-independent | Standard in diffusion (32 groups typical) |
| **Residual Connections** | y = F(x) + x; gradient highway; 1x1 projection for dim change | Every block in the U-Net |

**Next module:** Attention mechanisms -- the other key ingredient in modern diffusion architectures.